# Preprocessing Features Utilities Demo

Demostración interactiva de las transformaciones e ingeniería de variables temporales implementadas en `_preprocessing_features.py` y expuestas por `ds_utils.preprocessing.preprocessing_utilities`.

Funciones cubiertas en este notebook:
- `resample_time`
- `prepare_time_features`

## Imports

In [1]:
# RECARGA AUTOMÁTICA DE MÓDULOS EXTERNOS
%load_ext autoreload
%autoreload 2

import pandas as pd

from ds_utils.preprocessing.preprocessing_utilities import (
    resample_time,
    prepare_time_features,
)

## resample_time

`resample_time` remuestrea una serie temporal a una frecuencia regular fija (`freq`), insertando timestamps ausentes y conservando observaciones originales (`asfreq`) sin agregaciones forzadas, con opciones de redondeo (`alignment='round'`) e imputación (`ffill`/`bfill`).

In [2]:
# 1. Serie temporal con desfase (jitter) y un hueco entre 10:05 y 10:20
df_resample_raw = pd.DataFrame({
    "timestamp": [
        "2024-01-01 10:00:04",
        "2024-01-01 10:05:02",
        # faltan 10:10 y 10:15
        "2024-01-01 10:20:00",
        "2024-01-01 10:25:05",
    ],
    "power": [120.5, 122.0, 130.0, 131.2]
})

# 2. Remuestreo a 5 minutos con forward-fill
df_resampled = resample_time(
    df=df_resample_raw,
    col_time="timestamp",
    freq="5min",
    alignment="round",
    fill_method="ffill",
    fill_limit=2
)

print("Serie remuestreada a intervalos regulares de 5 minutos:")
display(df_resampled)

Serie remuestreada a intervalos regulares de 5 minutos:


,timestamp,power
0,2024-01-01 10:00:00,120.5
1,2024-01-01 10:05:00,122.0
2,2024-01-01 10:10:00,122.0
3,2024-01-01 10:15:00,122.0
4,2024-01-01 10:20:00,130.0
5,2024-01-01 10:25:00,131.2


## prepare_time_features

`prepare_time_features` convierte la columna temporal a datetime y genera automáticamente características temporales para ML (hora lineal, hora cíclica en seno/coseno, día de la semana numérico, textual y cíclico).

In [4]:
# 1. DataFrame con marcas temporales variadas
df_time_sample = pd.DataFrame({
    "timestamp": [
        "2024-01-01 00:00:00",  # Lunes medianoche
        "2024-01-01 06:00:00",  # Lunes mañana
        "2024-01-01 12:00:00",  # Lunes mediodía
        "2024-01-06 14:00:00",  # Sábado tarde
        "2024-01-07 20:00:00",  # Domingo noche
    ],
    "load": [50, 75, 120, 40, 35]
})

# 2. Generación completa de variables temporales lineales y cíclicas
df_time_features = prepare_time_features(
    df=df_time_sample,
    col_time="timestamp",
    hour_linear=True,
    hour_cyclic=True,
    weekday_linear="both",
    weekday_cyclic=True
)

print("DataFrame con variables temporales calculadas:")
display(df_time_features)

DataFrame con variables temporales calculadas:


,timestamp,load,hour,hour_sin,hour_cos,weekday,weekday_text,week_sin,week_cos
0,2024-01-01 00:00:00+00:00,50,0,0.000000e+00,1.000000e+00,0,Monday,0.000000,1.000000
1,2024-01-01 06:00:00+00:00,75,6,1.000000e+00,6.123234e-17,0,Monday,0.000000,1.000000
2,2024-01-01 12:00:00+00:00,120,12,1.224647e-16,-1.000000e+00,0,Monday,0.000000,1.000000
3,2024-01-06 14:00:00+00:00,40,14,-5.000000e-01,-8.660254e-01,5,Saturday,-0.974928,-0.222521
4,2024-01-07 20:00:00+00:00,35,20,-8.660254e-01,5.000000e-01,6,Sunday,-0.781831,0.623490
